# KSPP pseudopotentials

Set `ksppresolver=True` and QEpy resolves pseudopotentials automatically from [KSPP](https://github.com/Quantum-MultiScale/KSPP). No need to set `pseudo_dir` or `atomic_species` by hand.

## Available KSPP tables

Pass a `table` path to `ksppresolver` (or to `QEInput` / `Driver` / `QEpyCalculator`). Default: `ultrasoft/gbrv-v1.5`.

| Type | `table=` value | Description |
| --- | --- | --- |
| GBRV ultrasoft | `ultrasoft/gbrv-v1.5` | GBRV Vanderbilt ultrasoft (default) |
| ONCV norm-conserving | `norm-conserving/nc-sr-05` | PseudoDojo ONCVPSP v0.5 |
| ONCV norm-conserving | `norm-conserving/nc-sr-04` | PseudoDojo ONCVPSP v0.4.1 (reference) |
| ONCV norm-conserving | `norm-conserving/nc-fr-04` | PseudoDojo ONCVPSP v0.4, fully relativistic / SOC |
| ONCV norm-conserving | `norm-conserving/nc-sr-04-3plus` | PseudoDojo ONCVPSP, Ln³⁺ lanthanides |
| ONCV norm-conserving | `norm-conserving/nc-sr` | PseudoDojo ONCVPSP v0.3 (legacy) |
| JTH PAW | `paw/jth-v1.1` | JTH PAW v1.1 (PseudoDojo) |
| JTH PAW | `paw/jth-v1.0` | JTH PAW v1.0 (PseudoDojo) |
| GBRV PAW | `paw/gbrv-v1.5` | GBRV PAW v1.5 |

**Examples in this notebook**

1. **Driver** SCF on FCC Al (default GBRV ultrasoft)
2. **QEpyCalculator** SCF on rocksalt NaCl
3. Same systems with ONCV and JTH PAW via `table=...`

Requires a working QEpy + QE installation and network access on first use (pseudos are cached locally afterward).

In [8]:
from pathlib import Path

from ase.build import bulk
from qepy.calculator import QEpyCalculator
from qepy.driver import Driver
from qepy.io import QEInput

NOTEBOOK_DIR = Path(".").resolve()

## SCF with `Driver` (Aluminum)

Default library: GBRV ultrasoft (`ultrasoft/gbrv-v1.5`).

In [9]:
atoms_al = bulk("Al", "fcc", a=4.05, cubic=True)

qe_options_al = {
    "&control": {"calculation": "'scf'"},
    "&system": {
        "ibrav": 0,
        "degauss": 0.005,
        "ecutwfc": 30,
        "occupations": "'smearing'",
    },
    "&electrons": {"mixing_beta": 0.5},
    "k_points gamma": [],
}

pwin = QEInput(qe_options=qe_options_al, atoms=atoms_al, ksppresolver=True)
driver = Driver(pwin, atoms=atoms_al, logfile=NOTEBOOK_DIR / "al_scf_gbrv.out")
driver.scf()

if driver.is_root:
    print("PP type: GBRV ultrasoft")
    print("atomic_species:", pwin.qe_options["atomic_species"])
    print("converged:", driver.check_convergence())
    print("energy (Ry):", driver.get_energy())

driver.stop()

PP type: GBRV ultrasoft
atomic_species: ['Al    26.981538 al_pbe_v1.uspp.F.UPF']
converged: True
energy (Ry): -26.33516667403749


## SCF with `QEpyCalculator` (NaCl)

Same default GBRV ultrasoft library.

In [10]:
atoms_nacl = bulk("NaCl", "rocksalt", a=5.64, cubic=True)

qe_options_nacl = {
    "&control": {"calculation": "'scf'"},
    "&system": {
        "ibrav": 0,
        "degauss": 0.02,
        "ecutwfc": 40,
        "occupations": "'smearing'",
    },
    "&electrons": {"mixing_beta": 0.5},
    "k_points gamma": [],
}

atoms_nacl.calc = QEpyCalculator(
    atoms=atoms_nacl,
    qe_options=qe_options_nacl,
    ksppresolver=True,
    logfile=NOTEBOOK_DIR / "nacl_scf_gbrv.out",
)

energy = atoms_nacl.get_potential_energy()

if atoms_nacl.calc.is_root:
    print("PP type: GBRV ultrasoft")
    print("atomic_species:", atoms_nacl.calc.qe_options["atomic_species"])
    print("energy (Ry):", energy)

PP type: GBRV ultrasoft
atomic_species: ['Na    22.989769 na_pbe_v1.5.uspp.F.UPF', 'Cl    35.450000 cl_pbe_v1.4.uspp.F.UPF']
energy (Ry): -7013.1433612556475


## ONCV norm-conserving (PseudoDojo)

Pass `table="norm-conserving/nc-sr-04"` to use the ONCV library from KSPP.

In [11]:
pwin = QEInput(
    qe_options=qe_options_al,
    atoms=atoms_al,
    ksppresolver=True,
    table="norm-conserving/nc-sr-04",
)
driver = Driver(pwin, atoms=atoms_al, logfile=NOTEBOOK_DIR / "al_scf_oncv.out")
driver.scf()

if driver.is_root:
    print("PP type: ONCV norm-conserving")
    print("atomic_species:", pwin.qe_options["atomic_species"])
    print("energy (Ry):", driver.get_energy())

driver.stop()

atoms_nacl.calc = QEpyCalculator(
    atoms=atoms_nacl,
    qe_options=qe_options_nacl,
    ksppresolver=True,
    table="norm-conserving/nc-sr-04",
    logfile=NOTEBOOK_DIR / "nacl_scf_oncv.out",
)
energy = atoms_nacl.get_potential_energy()

if atoms_nacl.calc.is_root:
    print("PP type: ONCV norm-conserving")
    print("atomic_species:", atoms_nacl.calc.qe_options["atomic_species"])
    print("energy (Ry):", energy)

PP type: ONCV norm-conserving
atomic_species: ['Al    26.981538 Al.upf']
energy (Ry): -18.04192374981093
PP type: ONCV norm-conserving
atomic_species: ['Na    22.989769 na_pbe_v1.5.uspp.F.UPF', 'Cl    35.450000 cl_pbe_v1.4.uspp.F.UPF']
energy (Ry): -7013.143361280581


## JTH PAW (PseudoDojo)

Pass `table="paw/jth-v1.1"` to use the JTH PAW library from KSPP.

In [12]:
pwin = QEInput(
    qe_options=qe_options_al,
    atoms=atoms_al,
    ksppresolver=True,
    table="paw/jth-v1.1",
)
driver = Driver(pwin, atoms=atoms_al, logfile=NOTEBOOK_DIR / "al_scf_jth.out")
driver.scf()

if driver.is_root:
    print("PP type: JTH PAW")
    print("atomic_species:", pwin.qe_options["atomic_species"])
    print("energy (Ry):", driver.get_energy())

driver.stop()

atoms_nacl.calc = QEpyCalculator(
    atoms=atoms_nacl,
    qe_options=qe_options_nacl,
    ksppresolver=True,
    table="paw/jth-v1.1",
    logfile=NOTEBOOK_DIR / "nacl_scf_jth.out",
)
energy = atoms_nacl.get_potential_energy()

if atoms_nacl.calc.is_root:
    print("PP type: JTH PAW")
    print("atomic_species:", atoms_nacl.calc.qe_options["atomic_species"])
    print("energy (Ry):", energy)

PP type: JTH PAW
atomic_species: ['Al    26.981538 Al.upf']
energy (Ry): -157.51075088366727
PP type: JTH PAW
atomic_species: ['Na    22.989769 na_pbe_v1.5.uspp.F.UPF', 'Cl    35.450000 cl_pbe_v1.4.uspp.F.UPF']
energy (Ry): -7013.143361209016
